In [ ]:
import importlib.metadata
import os
import subprocess
import sys
import tempfile
from pathlib import Path

if "tensorflow" in sys.modules and os.environ.get("TF_USE_LEGACY_KERAS") != "1":
    raise RuntimeError(
        "Restart the Colab session, then run this notebook from its first cell. "
        "NIF requires the legacy Keras setting before TensorFlow is imported."
    )
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

COLAB_REQUIREMENTS = [
    "tensorflow==2.21.0",
    "tf-keras==2.21.0",
    "keras==3.13.2",
    "tensorflow-probability==0.25.0",
    "tensorflow-model-optimization==0.8.1",
    "numpy==2.1.3",
    "scipy==1.16.3",
    "matplotlib==3.10.0",
    "tensorboard==2.21.0",
    "absl-py==1.4.0",
]
NIF_COMMIT = "ad9113df14590189b12068b6b3fa7c8c8d9187d4"

if sys.version_info < (3, 11) or sys.version_info >= (3, 14):
    raise RuntimeError(
        "Use a Colab Python 3.11–3.13 runtime for this pinned environment."
    )

missing = []
for requirement in COLAB_REQUIREMENTS:
    package, expected = requirement.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(requirement)
if missing:
    package_modules = {
        "tensorflow": "tensorflow",
        "tf-keras": "tf_keras",
        "keras": "keras",
        "numpy": "numpy",
        "scipy": "scipy",
        "matplotlib": "matplotlib",
        "tensorflow-probability": "tensorflow_probability",
        "tensorflow-model-optimization": "tensorflow_model_optimization",
        "tensorboard": "tensorboard",
        "absl-py": "absl",
    }
    changing = {item.split("==")[0] for item in missing}
    loaded = [
        module
        for package, module in package_modules.items()
        if package in changing and module in sys.modules
    ]
    if loaded:
        raise RuntimeError(
            "Restart the Colab session before changing packages; already imported: "
            + ", ".join(loaded)
        )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *COLAB_REQUIREMENTS,
        ]
    )

runtime_root = Path(tempfile.gettempdir()) / ("nif-tutorials-v2-" + NIF_COMMIT[:12])
stamp = runtime_root / ".nif-source-commit"
if not stamp.exists() or stamp.read_text().strip() != NIF_COMMIT:
    runtime_root.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            "--upgrade",
            "--disable-pip-version-check",
            "--target",
            str(runtime_root),
            "git+https://github.com/pswpswpsw/nif.git@" + NIF_COMMIT,
        ]
    )
    stamp.write_text(NIF_COMMIT)


def replace_compatibility(path, old, new):
    content = path.read_text(encoding="utf-8")
    if new in content:
        return
    if old in content:
        path.write_text(content.replace(old, new), encoding="utf-8")
    else:
        raise RuntimeError("Pinned NIF source changed unexpectedly: " + str(path))


optimizer_file = runtime_root / "nif" / "optimizers" / "external_optimizers.py"
replace_compatibility(
    optimizer_file,
    "from tensorflow.python.distribute import distribution_strategy_context as distribute_ctx",
    "from tensorflow.python.distribute import distribute_lib as distribute_ctx",
)
for optimizer_class in ("AdaBeliefOptimizer", "L4Adam"):
    replace_compatibility(
        optimizer_file,
        "class " + optimizer_class + "(tf.keras.optimizers.Optimizer):",
        "class " + optimizer_class + "(tf.keras.optimizers.legacy.Optimizer):",
    )
for name in ("lbfgs.py", "lbfgs_V2.py"):
    path = runtime_root / "nif" / "optimizers" / name
    content = path.read_text(encoding="utf-8")
    if "np.product(" in content:
        path.write_text(content.replace("np.product(", "np.prod("), encoding="utf-8")

tfr_file = runtime_root / "nif" / "data" / "tfr_dataset.py"
replace_compatibility(
    tfr_file,
    "target = tf.transpose(\n"
    "                batch_file[self.n_feature : self.n_feature + self.n_target]\n            )",
    "target = tf.transpose(\n"
    "                tf.squeeze(tf.stack("
    "batch_file[self.n_feature : self.n_feature + self.n_target], 0), 1)\n"
    "            )",
)
replace_compatibility(
    tfr_file,
    "target = tf.transpose(batch_file[-self.n_target :])",
    "target = tf.transpose(\n"
    "                tf.squeeze(tf.stack(batch_file[-self.n_target :], 0), 1)\n            )",
)

gc_file = runtime_root / "nif" / "optimizers" / "gtcf.py"
replace_compatibility(
    gc_file,
    "    def get_centralized_gradients_for_optimizer(loss, params):\n",
    """    if not getattr(optimizer, "_nif_gradient_centralization", False):
        original_get_gradients = optimizer._get_gradients

        def centralized_tape_gradients(*args, **kwargs):
            gradients_and_variables = original_get_gradients(*args, **kwargs)
            return [
                (gradient - tf.reduce_mean(
                    gradient, axis=list(range(len(gradient.shape) - 1)), keepdims=True
                )
                if gradient is not None and len(gradient.shape) > 1 else gradient, variable)
                for gradient, variable in gradients_and_variables
            ]

        optimizer._get_gradients = centralized_tape_gradients
        optimizer._nif_gradient_centralization = True

    def get_centralized_gradients_for_optimizer(loss, params):
""",
)

if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
if "nif" in sys.modules:
    loaded_nif = Path(sys.modules["nif"].__file__).resolve()
    if not loaded_nif.is_relative_to(runtime_root.resolve()):
        raise RuntimeError(
            "Restart the session: an unrelated NIF installation is already imported."
        )
print("Python:", sys.version.split()[0])
for requirement in COLAB_REQUIREMENTS:
    package = requirement.split("==")[0]
    print(package + ":", importlib.metadata.version(package))
print("NIF source commit:", NIF_COMMIT)
print("Compatibility copy:", runtime_root / "nif")


In [ ]:
import tensorflow as tf
import nif
import numpy as np
import time
import logging
import contextlib
from matplotlib import pyplot as plt
from mpl_toolkits import mplot3d
from nif.optimizers import centralized_gradients_for_optimizer

# A simple flow over a cylinder dataset from Castro, an AMR solver

In [ ]:
from nif.demo import CylinderFlow
tmp = CylinderFlow()
train_data = tmp.data
sample_weight = tmp.sample_weight

In [ ]:
print(train_data.shape)
print(sample_weight.shape)
print(tmp.std)

# Try NIFMultiScale linear

In [ ]:
cfg_shape_net = {
    "use_resblock":False,
    "connectivity": 'last_layer',
    "input_dim": 2,
    "output_dim": 2,
    "units": 30,
    "nlayers": 2,
    "weight_init_factor": 0.01,
    "omega_0":30.0
}
cfg_parameter_net = {
    "use_resblock":False,
    "input_dim": 1,
    "latent_dim": 10,
    "units": 30,
    "nlayers": 2,
    "activation": 'swish'
}

In [ ]:
enable_multi_gpu = False
enable_mixed_precision = False
nepoch = 600
lr = 1e-3
batch_size = 1024
checkpt_epoch = 1000
display_epoch = 100
print_figure_epoch = 100
# get training data set
num_total_data = train_data.shape[0]
dataset_inputs = tf.data.Dataset.from_tensor_slices((train_data[:, :3]))
dataset_outputs = tf.data.Dataset.from_tensor_slices(train_data[:, 3:5])
dataset_weight = tf.data.Dataset.from_tensor_slices(sample_weight)
train_dataset = tf.data.Dataset.zip((dataset_inputs, dataset_outputs, dataset_weight))
train_dataset = train_dataset.shuffle(num_total_data).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)
# mixed precision?
if enable_mixed_precision:
    mixed_policy = "mixed_float16"
    # we might need this for `model.fit` to automatically do loss scaling
    policy = nif.mixed_precision.Policy(mixed_policy)
    nif.mixed_precision.set_global_policy(policy)
else:
    mixed_policy = 'float32'
def scheduler(epoch, lr):
    if epoch < 200:
        return lr
    elif epoch < 400:
        return 5e-4
    elif epoch < 600:
        return 2e-4
    else:
        return 3e-5
scheduler_callback = tf.keras.callbacks.LearningRateScheduler(scheduler)

class LossAndErrorPrintingCallback(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.train_begin_time = time.time()
        self.history_loss = []
        logging.basicConfig(filename='./log_multiscale', level=logging.INFO, format='%(message)s')

    def on_epoch_begin(self, epoch, logs=None):
        self.ts = time.time()

    def on_epoch_end(self, epoch, logs=None):
        if epoch % display_epoch == 0:
            tnow = time.time()
            te = tnow - self.ts
            logging.info("Epoch {:6d}: avg.loss pe = {:4.3e}, {:d} points/sec, time elapsed = {:4.3f} hours".format(
                epoch, logs['loss'], int(batch_size / te), (tnow - self.train_begin_time) / 3600.0))
            self.history_loss.append(logs['loss'])
        if epoch % print_figure_epoch == 0:
            plt.figure()
            plt.semilogy(self.history_loss)
            plt.xlabel('epoch: per {} epochs'.format(print_figure_epoch))
            plt.ylabel('MSE loss')
            plt.savefig('./loss_multiscale.png')
            plt.close()

        if epoch % checkpt_epoch == 0 or epoch == nepoch - 1:
            print('save checkpoint epoch: %d...' % epoch)
            self.model.save_weights("./saved_weights_multiscale/ckpt-{}/ckpt".format(epoch))


In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="./tb-logs", update_freq='epoch')
%load_ext tensorboard
%tensorboard --logdir tb-logs

# Try `NIFMultiScale`

In [ ]:
cm = tf.distribute.MirroredStrategy().scope() if enable_multi_gpu else contextlib.nullcontext()
with cm:
    optimizer = nif.optimizers.AdaBeliefOptimizer(lr)
    # optimizer = tf.keras.optimizers.Adam(lr)
    optimizer.get_gradients = centralized_gradients_for_optimizer(optimizer)
    loss_fun = tf.keras.losses.MeanSquaredError()
    model_ori = nif.NIFMultiScaleLastLayerParameterized(cfg_shape_net, cfg_parameter_net, mixed_policy)
    model = model_ori.build()
    model.compile(optimizer, loss_fun)

# callbacks = []
# callbacks = [LossAndErrorPrintingCallback(), scheduler_callback]
# callbacks = [tensorboard_callback, ]
callbacks = [tensorboard_callback, LossAndErrorPrintingCallback(), scheduler_callback]
model.fit(train_dataset, epochs=nepoch, batch_size=batch_size, 
          shuffle=False, verbose=2, callbacks=callbacks, 
          use_multiprocessing=True)

# draw latent representation

In [ ]:
model_p_to_lr = model_ori.model_p_to_lr()

In [ ]:
t_arr = np.linspace(-1,1,100)
lr_arr = model_p_to_lr(t_arr)

In [ ]:
plt.plot(t_arr, lr_arr)

# compute the reconstruction on training data

In [ ]:
input_pred = train_data[:,:3]
index_ = np.diff(input_pred[:,0],append=0)>0
time_jump_index = np.arange(input_pred.shape[0])[index_]+1

uv_pred = model.predict((input_pred[:,:3]))

## we can do a sanity check to see that the number of grid point is changing over time, so it is an AMR dataset 

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(np.diff(time_jump_index),'k-o')
plt.xlabel('time',fontsize=20)
plt.ylabel('number of mesh point at that time',fontsize=20)

# draw the reconstruction

In [ ]:
mean = tmp.mean
std = tmp.std

ti = 0
ti_ = time_jump_index[ti]

fig,axs=plt.subplots(2,2,figsize=(10.75,8))
circle1 = plt.Circle((0, 0), 0.0035, color='grey')
circle2 = plt.Circle((0, 0), 0.0035, color='grey')
circle3 = plt.Circle((0, 0), 0.0035, color='grey')
circle4 = plt.Circle((0, 0), 0.0035, color='grey')


umax = np.max(train_data[:ti_,3]*std[3]+mean[3])
umin = np.min(train_data[:ti_,3]*std[3]+mean[3])
vmax = np.max(train_data[:ti_,4]*std[4]+mean[4])
vmin = np.min(train_data[:ti_,4]*std[4]+mean[4])

axs[0,0].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     train_data[:ti_,3]*std[3]+mean[3],levels=50,vmin=umin,vmax=umax)
axs[0,1].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     train_data[:ti_,4]*std[4]+mean[4],levels=50,vmin=vmin,vmax=vmax)

axs[1,0].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     uv_pred[:ti_,0]*std[3]+mean[3],levels=50,vmin=umin,vmax=umax)
axs[1,1].tricontourf(train_data[:ti_,1]*std[1]+mean[1], train_data[:ti_,2]*std[2]+mean[2], 
                     uv_pred[:ti_,1]*std[4]+mean[4],levels=50,vmin=vmin,vmax=vmax)


for j in range(2):
    for i in range(2):
        axs[i,j].set_xlim([-0.01,0.04])
        axs[i,j].set_ylim([-0.02,0.02])
        axs[i,j].axis('equal')
    
axs[0,0].set_title('U - true',fontsize=20)
axs[0,1].set_title('V - true',fontsize=20)
axs[1,0].set_title('U - pred',fontsize=20)
axs[1,1].set_title('V - pred',fontsize=20)
axs[0,0].add_patch(circle1)
axs[0,1].add_patch(circle2)
axs[1,0].add_patch(circle3)
axs[1,1].add_patch(circle4)
fig.suptitle('demo of a very small NIF with 2 layers and 30 neurons', fontsize=30)

# Summary
- we learned how to learn a linear representation on spatial-temporal dynamics on arbitrary mesh 
- we demonstrated on a (shortened) flow over a cylinder case with a small neural network, just to give a flavour how it is trained and how does it look like